# Mood Melody Palette
ECS7022P Computational Creativity Project
Azucena Denise

Instructions:
Run all cells before running the Main function in the last cell, which includes the interactive interface

In [ ]:
# Mount the Google Drive at /content/drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## I. Install Dependencies and Import Libraries
Note: T4 GPU was used to run code

In [ ]:
!pip install miditoolkit torch scikit-learn torchaudio gradio numpy mido pandas pretty_midi matplotlib wget note_seq transformers miditok tqdm seaborn huggingface_hub muspy symusic


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split,Dataset, DataLoader
import numpy as np
import gradio as gr
import pandas as pd
import miditoolkit
from miditoolkit import MidiFile
import pretty_midi
import matplotlib.pyplot as plt
import matplotlib.patches as patch
import symusic
from tqdm import tqdm
import muspy
import os
import miditok
from miditok import CPWord, TokenizerConfig
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
from transformers.models.auto.tokenization_auto import AutoTokenizer
from transformers.data.data_collator import DataCollatorForLanguageModeling
from transformers.trainer_callback import EarlyStoppingCallback
from transformers.trainer import Trainer
from transformers.trainer_callback import TrainerCallback
from transformers.training_args import TrainingArguments
import json
import wget
from IPython.display import Audio
import mido
import time

## II. Dataset Preparation
### Note: To download GiantMIDI and EMOPIA Dataset
The owner has set permission for users to have limited access. Downloading via !gdown does not work and the whole dataset is not in the github repository although they mentioned about downloading in the "dislaimer" regarding downloading the dataset

Please download from this drive, unzip and upload the MIDI contents to the '/content/drive/MyDrive/GiantMIDI_files' folder after mounting on google drive:

https://drive.google.com/drive/folders/1Stz3CAvMoplo79LR5I3onMWRelCugBYS



#### Download and upload GiantMIDI:
Only run once

In [ ]:
!mkdir -p /content/drive/MyDrive/GiantMIDI_files
!unzip -qq "/content/drive/MyDrive/midis_v1.2.zip" \
        -d "/content/drive/MyDrive/GiantMIDI_files"

#### Run for processing GiantMIDI data

In [ ]:
def process_giant_midi_dataset(giant_dir, output_dir, tokeniser):
    ''' Process GiantMIDI-Piano dataset and estimate arousal-valence values'''
    giant_files = []
    giant_metadata = []

    # Find all MIDI files in the GiantMIDI directory
    for file in tqdm(os.listdir(giant_dir), desc="Finding GiantMIDI files"):
        if file.endswith(".mid"):
            giant_files.append(os.path.join(giant_dir, file))

    print(f"Found {len(giant_files)} MIDI files in GiantMIDI-Piano dataset")

    # Process each GiantMIDI file
    for midi_file in tqdm(giant_files, desc="Processing GiantMIDI files"):
        try:
            # Generate a unique ID for this file
            file_id = os.path.splitext(os.path.basename(midi_file))[0]

            # Estimate arousal and valence values
            arousal, valence = giantmidi_av(midi_file)

            # Create output path
            output_file = os.path.join(output_dir, f"giant_{file_id}.json")

            # Process the MIDI file
            processed = preprocess_midi(midi_file, output_file, tokeniser)

            if processed:
                # Add to metadata
                giant_metadata.append({
                    "YouTube_ID": f"giant_{file_id}",  # Use a prefix to distinguish from EMOPIA
                    "seg_id": 0,  # GiantMIDI doesn't have segments
                    "arousal": arousal,
                    "valence": valence,
                    "source": "GiantMIDI-Piano"
                })
        except Exception as e:
            print(f"Error processing {midi_file}: {str(e)}")

    return giant_metadata

def giantmidi_av(midi_path):
        '''Define arousal-valence estimation function'''
        # Load MIDI file
        midi_data = pretty_midi.PrettyMIDI(midi_path)

        # Extract features that correlate with arousal and valence
        tempo = midi_data.get_tempo_changes()[1]  # tempo: faster tempos often correlate with higher arousal
        if len(tempo) > 0:
            avg_tempo = np.mean(tempo)
        else:
            avg_tempo = 120  # Default tempo

        #  More notes equals to higher arousal
        total_notes = sum(len(instrument.notes) for instrument in midi_data.instruments)
        duration = midi_data.get_end_time()
        density = total_notes / duration if duration > 0 else 0

        # Higher velocity often correlates with higher arousal
        all_velocities = [note.velocity for instrument in midi_data.instruments
                         for note in instrument.notes]
        avg_velocity = np.mean(all_velocities) if all_velocities else 64

        # More pitch variation can correlate with higher valence
        all_pitches = [note.pitch for instrument in midi_data.instruments
                      for note in instrument.notes]
        pitch_variance = np.var(all_pitches) if len(all_pitches) > 1 else 0

        # Major/minor analysis
        key_analysis = midi_data.key_signature_changes
        is_major = False
        if key_analysis:
            is_major = key_analysis[0].key_number % 2 == 0  # Even numbers are major keys

        # Convert features to arousal and valence estimates
        norm_tempo = max(0, min(1, float((avg_tempo - 40) / 160))) # Normalise tempo between 0-1 (assume range 40-200 BPM)

        # Normalise density (assume max density of 10 notes per second)
        norm_density = max(0, min(1, density / 10))

        # Normalise velocity (already in range 0-127)
        norm_velocity = avg_velocity / 127

        # Normalise pitch variance (assume max variance of 500)
        norm_pitch_var = max(0, min(1,float(pitch_variance / 500)))

        # Calculate arousal (influenced by tempo, density, and velocity)
        arousal = 0.4 * norm_tempo + 0.3 * norm_density + 0.3 * norm_velocity

        # Calculate valence (influenced by pitch variance and major/minor)
        valence = 0.6 * norm_pitch_var + (0.4 if is_major else 0.2)

        # Ensure values are between 0 and 1
        arousal = max(0, min(1, float(arousal)))
        valence = max(0, min(1, float(valence)))

        return arousal, valence


### Pre-rocess and Tokenise EMOPIA and GiantMIDI MIDI Files

In [ ]:
def preprocess_midi(midi_path, output_path, tokeniser):
    '''Convert MIDI files to tokenized format using MIDItok's CPWord.'''
    try:
        # Load MIDI file
        midi = symusic.Score(midi_path)

        # Tokenize the MIDI file
        tokens = tokeniser(midi)  # This returns a TokSequence object

        # Flattens tokens
        if hasattr(tokens, 'ids'):
            token_ids = tokens.ids
        else:
            token_ids = [token for track in tokens for token in track.ids]

        # Create directory for output if it doesn't exist
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Use the tokenizer's built-in save_tokens method
        with open(output_path, 'w') as f:
            json.dump({'tokens': token_ids, 'original_path': midi_path}, f)


        return True

    except Exception as e:
        print(f"Error processing {midi_path}: {str(e)}")
        return False


# Load EMOPIA Dataset
emopia = muspy.EMOPIADataset("emopia/", download_and_extract=True)
emopia.convert()
annotations_data = []

for music in emopia:
    for ann in music.annotations:
        annotations_data.append(ann.annotation)  # Each is a dict with 'emo_class', 'YouTube_ID', 'seg_id'

# Create dataframe
df = pd.DataFrame(annotations_data)

# Add arousal and valence to
emotion_to_av = {
        '1': (0.75, 0.75),
        '2': (0.75, 0.25),
        '3': (0.25, 0.25),
        '4': (0.25, 0.75)}

def map_emotion_to_av(emotion):
    return pd.Series(emotion_to_av.get(str(emotion), (None, None)))

df[['arousal', 'valence']] = df['emo_class'].apply(map_emotion_to_av)
df.to_csv('emopia_with_av.csv') # Save to CSV

# Create tokenizer configuration
config = TokenizerConfig(
    pitch_range=(21, 109),
    beat_res={(0, 4): 8, (4, 12): 4},
    num_velocities=32,
    use_programs=False,
    use_chords = True,
    use_tempos= True,
    use_rests=True)

config.one_token_stream_for_programs = False
# Initialise tokenizer with configuration
tokeniser = CPWord(tokenizer_config=config)

# Process all MIDI files
#===============EMOPIA========================#
emopia_midi_path = '/content/emopia/EMOPIA_2.2/midis'
output_dir = '/content/midi_out'


midi_files = []
# Find all Emopia MIDI files
for root, dirs, files in os.walk(emopia_midi_path):
    for file in files:
        if file.endswith('.mid'):
            midi_files.append(os.path.join(root, file))

# Process each Emopia file
for midi_file in tqdm(midi_files):
    # Create output path with the same structure but different extension
    rel_path = os.path.relpath(midi_file, emopia_midi_path)
    output_file = os.path.join(output_dir, rel_path.replace('.mid', '.json'))

    # Process the file
    processed = preprocess_midi(midi_file, output_file, tokeniser)
    if not processed:
        print(f"Failed to process {midi_file}")

#===============GiantMidi========================#

giant_dir = "/content/drive/MyDrive/GiantMIDI_files/midis"
giant_metadata = process_giant_midi_dataset(giant_dir, output_dir, tokeniser)

# Create a DataFrame for GiantMIDI metadata
giant_df = pd.DataFrame(giant_metadata)

# Combine both datasets' metadata
combined_df = pd.concat([df, giant_df], ignore_index=True)

# Save the combined metadata
combined_df.to_csv('combined_dataset_with_av.csv', index=False)

# Save tokenizer params
tokeniser.save('cpword_params.json')

print(f"Processed {len(df)} EMOPIA samples and {len(giant_df)} GiantMIDI samples")
print(f"Total dataset size: {len(combined_df)} samples")


62988951552it [00:03, 18688305025.89it/s]


Successfully downloaded source : /content/emopia/EMOPIA_2.2.zip .
Extracting archive : /content/emopia/EMOPIA_2.2.zip ...
Successfully extracted archive : /content/emopia .
Converting and saving the dataset...


100%|██████████| 1071/1071 [00:05<00:00, 190.50it/s]


Successfully saved 1071 out of 1071 files.


<ipython-input-5-97170124a5c9>:68: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  tokeniser = CPWord(tokenizer_config=config)
Finding GiantMIDI files: 100%|██████████| 10855/10855 [00:00<00:00, 927312.11it/s]


Found 10855 MIDI files in GiantMIDI-Piano dataset


Processing GiantMIDI files: 100%|██████████| 10855/10855 [1:06:50<00:00,  2.71it/s]

Processed 1071 EMOPIA samples and 10855 GiantMIDI samples
Total dataset size: 11926 samples


## III.Using MuPT (Pre-trained Model)

### Load Pre-trained Model

In [ ]:
# Load the pre-trained MuPT model
model_path = 'm-a-p/MuPT_v1_8192_110M'
tokenizer_mupt = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True, use_fast= False)
model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True).to('cuda').half().eval()
model = model.float()
# Verify the model is on CUDA
print(f"Model is on CUDA: {next(model.parameters()).is_cuda}")

In [ ]:
class CombinedDataset(Dataset):
    def __init__(self, metadata_csv, tokens_dir, tokeniser, max_length=512):
        self.metadata = pd.read_csv(metadata_csv)
        self.tokens_dir = tokens_dir
        self.tokeniser = tokeniser
        self.max_length = max_length
        # Use CPU as the default device for dataset loading
        self.device = "cpu"

        # Create a mapping of YouTube_ID and seg_id to tokenized file paths
        self.token_file_map = {}
        for file_name in os.listdir(tokens_dir):
            if file_name.endswith('.json'):
                self.token_file_map[file_name] = os.path.join(tokens_dir, file_name)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, i):
        row = self.metadata.iloc[i]

        # Get YouTube ID and segment ID from the current row
        youtube_id = row['YouTube_ID']
        seg_id = row['seg_id']

        # Try different possible file naming patterns
        potential_file_names = [
            f"{youtube_id}_{seg_id}.json",
            f"{youtube_id}{seg_id}.json",
            f"{youtube_id}.json"
        ]

        # Try to find the matching file
        token_file = None
        for file_name in potential_file_names:
            if file_name in self.token_file_map:
                token_file = self.token_file_map[file_name]
                break

        # If not found by exact name, look for files containing the YouTube ID
        if token_file is None:
            for file_path in self.token_file_map.values():
                file_name = os.path.basename(file_path)
                if youtube_id in file_name:
                    token_file = file_path
                    break

        # If still not found, raise an error or use a default
        if token_file is None:
            print(f"Warning: No token file found for {youtube_id}_{seg_id}")
            # Create default/empty data - ALWAYS ON CPU
            empty_input = torch.zeros(10, dtype=torch.long)
            return {
                "input_ids": empty_input,  # CPU tensor
                "attention_mask": torch.ones_like(empty_input),  # CPU tensor
                "labels": empty_input  # CPU tensor
            }

        # Load the tokenised file
        with open(token_file, 'r') as f:
            token_data = json.load(f)

        # Get the arousal-valence values from the current row
        arousal = row['arousal']
        valence = row['valence']

        # Create conditioning prompt
        prompt = f"Generate a melody with arousal {arousal:.2f} and valence {valence:.2f}:"

        # Tokenise the prompt
        prompt_tokens = self.tokeniser.encode(prompt, add_special_tokens=False)

        flat_tokens = []
        for token_group in token_data['tokens']:
            if isinstance(token_group, list):
                flat_tokens.extend(token_group)  # Add all elements from sublist
            else:
                flat_tokens.append(token_group)  # Add single element

        # Combine prompt tokens with music tokens
        combined_tokens = prompt_tokens + flat_tokens

        # Truncate if needed
        if len(combined_tokens) > self.max_length:
            combined_tokens = combined_tokens[:self.max_length]

        # Create tensors on CPU only
        input_ids = torch.tensor(combined_tokens, dtype=torch.long)
        attention_mask = torch.ones(len(input_ids), dtype=torch.long)
        labels = input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

class CustomDataCollator(DataCollatorForLanguageModeling):
    '''Note: this was generated by Claude (Anthropic,2024) to properly mask padded tokens during training and prevent
    issues when managing padding'''
    def __init__(self, tokenizer, mlm=False, target_device="cuda"):
        super().__init__(tokenizer=tokenizer, mlm=mlm)
        self.target_device = target_device

    def __call__(self, features):
        # Get the maximum length in this batch
        max_length = max(len(feature["input_ids"]) for feature in features)

        # Pad all sequences to the max length for this batch
        for feature in features:
            padding_length = max_length - len(feature["input_ids"])

            if padding_length > 0:
                # Pad input_ids with zeros (or padding token)
                feature["input_ids"] = torch.cat([
                    feature["input_ids"],
                    torch.zeros(padding_length, dtype=torch.long)])

                # Pad attention_mask with zeros (masked positions)
                feature["attention_mask"] = torch.cat([
                    feature["attention_mask"],
                    torch.zeros(padding_length, dtype=torch.long)])

                # Pad labels with -100 (ignored in loss computation)
                feature["labels"] = torch.cat([
                    feature["labels"],
                    torch.full((padding_length,), -100, dtype=torch.long)])

        # Move everything to the target device (GPU) at this point
        for feature in features:
            for key in feature:
                feature[key] = feature[key].to(self.target_device)

        # Call the parent collator
        return super().__call__(features)

### Fine tune MuPT

In [ ]:
# Disable Weights and Biases logging
os.environ["WANDB_DISABLED"] = "true"

# Freeze all parameters
model.requires_grad_(False)

# Unfreeze the language modeling head and last layer
model.lm_head.requires_grad_(True)
for param in model.model.layers[-1].parameters():
    param.requires_grad = True

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

# Create dataset instance
full_dataset = CombinedDataset('combined_dataset_with_av.csv', '/content/midi_out', tokenizer_mupt)
dataset_size = len(full_dataset)
train_size = int(0.8 * dataset_size)
eval_size = dataset_size - train_size

# Split dataset into train and evaluation

train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size],
    generator=torch.Generator().manual_seed(42))

print(f"Training on {train_size} samples, evaluating on {eval_size} samples")

# Configure training
training_args = TrainingArguments(
    output_dir="./mupt-emotion-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=100,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False )

# Initialise components
data_collator = CustomDataCollator(tokenizer=tokenizer_mupt, mlm=False)
early_stopping = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)

# Initialise trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[early_stopping])

# Execute training
try:
    trainer.train()
    trainer.save_model("./mupt-emotion-finetuned-final")
    tokenizer_mupt.save_pretrained("./mupt-emotion-finetuned-final")
    print(f"Final evaluation results: {trainer.evaluate()}")
except Exception as e:
    print(f"Training failed: {str(e)}")
    # Diagnostic checks
    print("\nModel structure checks:")
    print(f"Model type: {type(model).__name__}")
    print(f"Has lm_head: {'lm_head' in dir(model)}")
    print(f"Layers count: {len(model.model.layers)}")

    # Test forward pass
    print("\nTesting forward pass:")
    try:
        with torch.no_grad():
            sample = eval_dataset[0]
            inputs = {k: v.unsqueeze(0).to(model.device) for k, v in sample.items()}
            model(**inputs)
        print("Forward pass successful")
    except Exception as f_e:
        print(f"Forward pass failed: {str(f_e)}")

Number of trainable parameters: 47,838,720
Training on 9540 samples, evaluating on 2386 samples


Step,Training Loss,Validation Loss
100,1.720500,1.054552
200,0.959200,0.907205
300,0.870600,0.869517
400,0.842700,0.825895
500,0.818300,0.802616
600,0.793900,0.787344
700,0.781700,0.774817
800,0.756900,0.762072
900,0.749200,0.755200
1000,0.749700,0.747791


Final evaluation results: {'eval_loss': 0.7477913498878479, 'eval_runtime': 139.6994, 'eval_samples_per_second': 17.08, 'eval_steps_per_second': 2.14, 'epoch': 0.4612159329140461}


## IV. Interactive Interface
1. Load fine-tuned model and tokeniser (loads original pre-trained model as a fallback)
2. Loads MIDI tokeniser for handling conversion between MIDI data and token sequences the model can work with
3. Creates an arousal-valence emotion map with 4 quadrants and respective emotions
4. Generates melody based on arousal/valence values
5. Backup in case neural generation fails
6. Converts midi to audio
7. Describes quadrant placements
8. Loads models, tokenisers and main interface

In [ ]:
# Path to the fine-tuned model
MODEL_PATH = "./mupt-emotion-finetuned-final"

def load_model_and_tokeniser():
    '''Load the fine-tuned MuPT model and tokeniser'''
    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        # Load the tokeniser
        tokeniser_stored = AutoTokenizer.from_pretrained(
            MODEL_PATH,
            trust_remote_code=True,
            use_fast=False)

        # Load the model
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            trust_remote_code=True).to(device)

        model.eval()  # Set to evaluation mode

        print("Model and tokeniser loaded successfully!")
        return model, tokeniser_stored

    except Exception as e:
        print(f"Error loading model: {str(e)}")

        # Try to load from the original MuPT path as fallback
        try:
            model_path = "m-a-p/MuPT_v1_8192_110M"
            tokeniser_mupt = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=False)
            model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True).to(device)
            model.eval()
            return model, tokeniser_mupt # Loads original MuPT as fallback
        except Exception as e2:
            print(f"Fallback also failed: {str(e2)}")
            return None, None

def load_cpword_tokeniser():
    '''Load the CPWord tokenizer for MIDI encoding/decoding '''
    if os.path.exists("cpword_params.json"):
        # Load saved tokeniser parameters
        tokeniser = CPWord(params="cpword_params.json")
        print("Loaded CPWord tokeniser from saved parameters")
    else:
        # Create new tokeniser with default configuration
        config = TokenizerConfig(
            pitch_range=(21, 109),
            beat_res={(0, 4): 8, (4, 12): 4},
            num_velocities=32,
            use_programs=False,
            use_chords=True,
            use_tempos=True,
            use_rests=True,
            one_token_stream=True
        )
        tokeniser = CPWord(tokenizer_config=config)
        print("Created new CPWord tokeniser")
    return tokeniser

def create_av_plot(valence, arousal):
    '''Create color plot for arousal/valence'''
    fig, ax = plt.subplots(figsize=(6, 6))

    # Continous color gradient plot
    x = np.linspace(0, 1, 100)
    y = np.linspace(0, 1, 100)
    X, Y = np.meshgrid(x, y)

    R = 0.7 * (1 - X) + 0.9 * Y
    G = 0.8 * X + 0.4 * Y
    B = 0.8 * (1 - Y) + 0.3 * X

    # Clip RGB values to be between 0 and 1
    R = np.clip(R, 0, 1)
    G = np.clip(G, 0, 1)
    B = np.clip(B, 0, 1)

    # Stack RGB channels
    RGB = np.dstack((R, G, B))

    # Plot the color gradient
    ax.imshow(RGB, origin='lower', extent=[0, 1, 0, 1], aspect='auto')

    # Add grid lines and axis labels
    ax.grid(color='white', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_xlabel('Valence (Negative → Positive)', fontsize=12)
    ax.set_ylabel('Arousal (Calm → Excited)', fontsize=12)
    ax.set_title('Emotion Space', fontsize=14, fontweight='bold')

    # Add quadrant labels with white text for visibility
    ax.text(0.25, 0.75, "Angry/Anxious", ha='center', va='center', fontweight='bold', color='white')
    ax.text(0.75, 0.75, "Excited/Happy", ha='center', va='center', fontweight='bold', color='white')
    ax.text(0.25, 0.25, "Calm/Sad", ha='center', va='center', fontweight='bold', color='white')
    ax.text(0.75, 0.25, "Calm/Happy", ha='center', va='center', fontweight='bold', color='white')

    # Draw crosshairs for center point
    ax.axhline(y=0.5, color='white', linestyle='-', alpha=0.5, linewidth=1)
    ax.axvline(x=0.5, color='white', linestyle='-', alpha=0.5, linewidth=1)

    # Plot the selected point
    ax.scatter(valence, arousal, color='white', s=150, zorder=5, edgecolor='black', linewidth=2)

    # Set limits
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Create a temporary file to save the plot
    plot_file = f"emotion_plot_{time.time()}.png"
    plt.tight_layout()
    plt.savefig(plot_file, dpi=100)
    plt.close()


    return plot_file


import os  # Make sure this is imported

def generate_melody(model, tokeniser, cpword_tokeniser, arousal, valence, key="C major", max_length=256, rhythm_pattern=None):
    '''Generate MIDI based on arousal and valence values'''
    # Create output directory
    output_dir = "/content/generated_midis"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Create a descriptive filename with valence and arousal values
    output_file = os.path.join(output_dir, f"v{valence:.2f}_a{arousal:.2f}_melody_{int(time.time())}.mid")

    # Create a conditioning prompt
    prompt = f"Generate a melody with arousal {arousal:.2f} and valence {valence:.2f}:"
    print(f"Generating with prompt: {prompt}")

    # Get device from model
    device = next(model.parameters()).device

    # Tokenise the prompt
    input_ids = tokeniser.encode(prompt, return_tensors="pt").to(device)

    try:
        # Generate the melody tokens
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_length=max_length,
                do_sample=True,
                top_p=0.92,
                repetition_penalty=1.1,
                num_return_sequences=1
            )

        # Extract generated tokens (remove prompt tokens)
        prompt_length = len(input_ids[0])
        generated_tokens = output[0][prompt_length:].cpu().tolist()

        # Convert to format expected by CPWord
        token_groups = []
        for i in range(0, len(generated_tokens), 5):
            if i + 5 <= len(generated_tokens):
                token_groups.append(generated_tokens[i:i+5])

        # Convert to MIDI
        score = cpword_tokeniser.decode(token_groups)

        # Apply key constraints
        midi_data = pretty_midi.PrettyMIDI()
        midi_data.instruments = score.instruments
        midi_data = enforce_key(midi_data, key)

        # Apply rhythm constraints if specified
        if rhythm_pattern:
            midi_data = enforce_rhythm(midi_data, rhythm_pattern)

        # Save the final MIDI file
        midi_data.write(output_file)
        print(f"Successfully generated MIDI file: {output_file}")

    except Exception as e:
        print(f"Error generating MIDI: {str(e)}")
        print("Using fallback MIDI generation...")
        # Create a simple MIDI file as fallback
        create_simple_midi(output_file, arousal, valence)

    return output_file

def create_simple_midi(output_path, arousal, valence):
    '''Create a simple MIDI file based on arousal and valence values (fallback)'''
    print("Creating fallback MIDI file...")
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)  # Piano

    # Map arousal to note density and velocity
    notes_per_bar = int(4 + arousal * 16)  # 4 to 20 notes per bar
    velocity = int(60 + arousal * 40)  # 60 to 100 velocity

    # Map valence to pitch range
    base_pitch = int(60 - (valence * 12) + (arousal * 6))  # Adjust pitch based on both dimensions
    pitch_range = int(7 + valence * 12)  # 7 to 19 semitones

    # Generate 8 bars of music
    duration = 16.0  # 4 bars at 120 BPM
    for i in range(notes_per_bar * 4):
        # Create note timing based on position
        start_time = i * (duration / (notes_per_bar * 4))

        # Note length varies with arousal (shorter for high arousal)
        note_length = (duration / (notes_per_bar * 4)) * (1.5 - arousal * 0.7)

        # Create a note
        note = pretty_midi.Note(
            velocity=velocity,
            pitch=base_pitch + np.random.randint(0, pitch_range),
            start=start_time,
            end=start_time + note_length)
        piano.notes.append(note)

    midi.instruments.append(piano)
    midi.write(output_path)
    print(f"Created fallback MIDI file: {output_path}")

    return output_path

def enforce_key(midi_data, key="C major"):
  '''Enforce a specific key on a MIDI file by mapping notes to the appropriate scale for added constraints'''
  key_mappings = {
    # Major scales
    "C major":        [0, 2, 4, 5, 7, 9, 11],
    "C♯ major":      [1, 3, 5, 6, 8, 10, 0],
    "D major":        [2, 4, 6, 7, 9, 11, 1],
    "E♭ major":      [3, 5, 7, 8, 10, 0, 2],
    "E major":        [4, 6, 8, 9, 11, 1, 3],
    "F major":        [5, 7, 9, 10, 0, 2, 4],
    "F♯ major":      [6, 8, 10, 11, 1, 3, 5],
    "G major":        [7, 9, 11, 0, 2, 4, 6],
    "A♭ major":      [8, 10, 0, 1, 3, 5, 7],
    "A major":        [9, 11, 1, 2, 4, 6, 8],
    "B♭ major":      [10, 0, 2, 3, 5, 7, 9],
    "B major":        [11, 1, 3, 4, 6, 8, 10],

    # Natural minor scales
    "A minor":        [9, 11, 0, 2, 4, 5, 7],
    "A♯ minor":      [10, 0, 1, 3, 5, 6, 8],
    "B minor":        [11, 1, 2, 4, 6, 7, 9],
    "C minor":        [0, 2, 3, 5, 7, 8, 10],
    "C♯ minor":      [1, 3, 4, 6, 8, 9, 11],
    "D minor":        [2, 4, 5, 7, 9, 10, 0],
    "E♭ minor":      [3, 5, 6, 8, 10, 11, 1],
    "E minor":        [4, 6, 7, 9, 11, 0, 2],
    "F minor":        [5, 7, 8, 10, 0, 1, 3],
    "F♯ minor":      [6, 8, 9, 11, 1, 2, 4],
    "G minor":        [7, 9, 10, 0, 2, 3, 5],
    "G♯ minor":      [8, 10, 11, 1, 3, 4, 6]}

  # Get the scale degrees for the key
  if key not in key_mappings:
      print(f"Key {key} not found in mappings, defaulting to C major")
      scale_degrees = key_mappings["C major"]
  else:
      scale_degrees = key_mappings[key]

  # For each track in the MIDI file
  for instrument in midi_data.instruments:
      for note in instrument.notes:
          # Get the note's pitch class (0-11, where 0 is C, 1 is C#, etc.)
          pitch_class = note.pitch % 12

          # If this pitch is not in our scale
          if pitch_class not in scale_degrees:
              # Find the nearest scale degree
              nearest_degree = min(scale_degrees,
                                    key=lambda x: min(abs(x - pitch_class),
                                                    12 - abs(x - pitch_class)))

              # Adjust the pitch to fit the key
              octave = note.pitch // 12
              note.pitch = (octave * 12) + nearest_degree

  return midi_data

def get_rhythm_patterns(pattern_name="default"):
    '''Return predefined rhythm patterns as fractions of a beat'''
    patterns = {
        "default": [0.25, 0.25, 0.25, 0.25],  # Quarter notes (regular)
        "waltz": [0.5, 0.25, 0.25],  # Waltz pattern (ONE-two-three)
        "march": [0.25, 0.125, 0.125, 0.25, 0.25],  # March rhythm
        "syncopated": [0.375, 0.125, 0.25, 0.25],  # Syncopated pattern
        "bossa_nova": [0.25, 0.125, 0.125, 0.25, 0.125, 0.125]  # Bossa nova rhythm
    }
    return patterns.get(pattern_name, patterns["default"])

def enforce_rhythm(midi_data, rhythm_pattern_name="default", strength=0.7):
    '''Apply a rhythmic pattern to a MIDI file'''

    if rhythm_pattern_name is None:
        return midi_data  # No rhythm enforcement

    # Get the rhythm pattern (note durations as fractions of a beat)
    pattern = get_rhythm_patterns(rhythm_pattern_name)
    pattern_length = sum(pattern)

    for instrument in midi_data.instruments:
        # Sort notes by start time
        instrument.notes.sort(key=lambda note: note.start)

        # Group notes into measures (assuming 4/4 time for simplicity)
        measure_length = 2.0  # 2 seconds per measure at 120 BPM
        measures = {}

        for note in instrument.notes:
            measure_idx = int(note.start / measure_length)
            if measure_idx not in measures:
                measures[measure_idx] = []
            measures[measure_idx].append(note)

        # Apply rhythm pattern to each measure
        for measure_idx, notes in measures.items():
            if not notes:
                continue

            # Calculate measure start time
            measure_start = measure_idx * measure_length

            # Apply rhythm pattern to notes in this measure
            pattern_position = 0
            pattern_idx = 0

            for note in notes:
                # Calculate where in the pattern this note should be
                relative_pos = (note.start - measure_start) / measure_length

                # Find the closest pattern position
                while pattern_position / pattern_length < relative_pos and pattern_idx < len(pattern) - 1:
                    pattern_position += pattern[pattern_idx]
                    pattern_idx = (pattern_idx + 1) % len(pattern)

                # Calculate target time based on pattern
                target_time = measure_start + (pattern_position / pattern_length) * measure_length

                # Move the note partially toward the target time
                note.start = note.start * (1 - strength) + target_time * strength

                # Adjust note length based on pattern duration
                pattern_duration = pattern[pattern_idx] * measure_length / pattern_length
                note.end = min(note.start + pattern_duration, note.end)

                # Minimum duration of note
                if note.end <= note.start:
                    note.end = note.start + 0.1

    return midi_data

def midi_to_audio(midi_file):
    '''Convert MIDI to audio for playback'''
    midi_data = pretty_midi.PrettyMIDI(midi_file)

    # Synthesise the MIDI to audio (returns a numpy array)
    audio_data = midi_data.synthesize(fs=44100)

    # Normalize audio (scale to -1.0 to 1.0 range)
    if np.max(np.abs(audio_data)) > 0:
        audio_data = audio_data / np.max(np.abs(audio_data))

    return (44100, audio_data)  #

def quadrant_description(valence, arousal):
    '''Return description of the emotional quadrant'''
    if arousal == 0.5 and valence == 0.5:
        return "Neutral/No Emotion"
    elif arousal > 0.5 and valence >= 0.5:
        return "Q1: Excited/Happy - Energetic, joyful music with upbeat tempo and bright harmonies"
    elif arousal >= 0.5 and valence < 0.5:
        return "Q2: Angry/Anxious - Intense, dissonant music with driving rhythms and tense harmonies"
    elif arousal < 0.5 and valence < 0.5:
        return "Q3: Calm/Sad - Slow, melancholic music with minor tonality and sparse texture"
    else:
        return "Q4: Calm/Happy - Peaceful, pleasant music with gentle rhythms and warm harmonies"

def build_mood_melody_interface():
    '''Gradio interface '''
    # Load the model, tokenisers
    # print("Loading model and tokenisers...")
    model, tokeniser = load_model_and_tokeniser()
    cpword_tokeniser = load_cpword_tokeniser()

    if model is None or tokeniser is None:
        print("ERROR: Could not load model or tokeniser. Interface may not work correctly.")

    def on_select_coordinate(x, y):
        """Handler for when coordinates are selected or changed"""
        # Convert x, y (-100 to 100) to valence, arousal (0 to 1)
        valence = (float(x) + 100) / 200
        arousal = (float(y) + 100) / 200

        # Create the plot visualization
        plot_path = create_av_plot(valence, arousal)

        # Generate MIDI
        midi_path = generate_melody(model, tokeniser, cpword_tokeniser, arousal, valence)

        # Get audio waveform from MIDI
        audio_data = midi_to_audio(midi_path)

        # Generate description
        quadrant_desc = quadrant_description(valence, arousal)

        return [plot_path,audio_data,midi_path, f"Valence: {valence:.2f}, Arousal: {arousal:.2f}",  quadrant_desc]

    # Build the Gradio interface
    with gr.Blocks(title="Mood Melody Palette") as demo:
        gr.Markdown(" Mood Melody Palette ")
        gr.Markdown("""
        Move the sliders to generate music that matches the selected mood.
        - **Valence**: Negative to positive emotion (left to right)
        - **Arousal**: Calm to excited (bottom to top) """)

        with gr.Row():
            # Left column - Controls
            with gr.Column(scale=1):
                with gr.Row():
                    x_slider = gr.Slider(
                        minimum=-100,
                        maximum=100,
                        value=0,
                        step=1,
                        label="Valence (Negative to Positive)")
                    y_slider = gr.Slider(
                        minimum=-100,
                        maximum=100,
                        value=0,
                        step=1,
                        label="Arousal (Calm to Excited)")

                generate_btn = gr.Button("Generate Music", variant="primary", size="lg")

                with gr.Row():
                    emotion_info = gr.Textbox(label="Position", interactive=False)
                    quadrant_info = gr.Textbox(label="Emotion Description", interactive=False)

            # Right column - Outputs
            with gr.Column(scale=1):
                plot_output = gr.Image(type="filepath", label="Emotion Map")

                with gr.Row():
                    # Use Gradio's audio component, passing it the sample rate and numpy array
                    audio_output = gr.Audio(label="Listen to the Generated Music", type="numpy")
                    file_output = gr.File(label="Download MIDI File")

        # Set up the event handlers
        generate_btn.click(
            on_select_coordinate,
            inputs=[x_slider, y_slider],
            outputs=[plot_output, audio_output, file_output, emotion_info, quadrant_info])

        # Update when sliders change
        x_slider.change(
            on_select_coordinate,
            inputs=[x_slider, y_slider],
            outputs=[plot_output, audio_output, file_output, emotion_info, quadrant_info])

        y_slider.change(
            on_select_coordinate,
            inputs=[x_slider, y_slider],
            outputs=[plot_output, audio_output, file_output, emotion_info, quadrant_info])

        demo.load(
            fn=on_select_coordinate,
            inputs=[gr.Number(value=0), gr.Number(value=0)],  # x=0, y=0 corresponds to valence=0.5, arousal=0.5
            outputs=[plot_output, audio_output, file_output, emotion_info, quadrant_info])

    return demo



# V. Main Function
After running all cells, run this for GUI


In [ ]:
# Main Function
if __name__ == "__main__":
    demo = build_mood_melody_interface()
    demo.launch(share=True)

Model and tokeniser loaded successfully!
Loaded CPWord tokeniser from saved parameters


<ipython-input-29-2904f577f73d>:42: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  tokeniser = CPWord(params="cpword_params.json")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7c7f496e339c8cad1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Reference
Anthropic. (2024). Claude 2 [Large language model]. Retrieved April 25, 2025, from https://claude.ai

### Evaluation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import entropy
import pretty_midi

# Configuration
EMOPIA_DIR = '/content/emopia/EMOPIA_2.2/midis'
GENERATED_DIR = '/content/generated_midis/'
SAMPLE_SIZE = 21

# Core Analysis Functions
def get_emotion_quadrant(valence, arousal):
    """Determine emotion quadrant from valence/arousal values"""
    if arousal >= 0.5 and valence >= 0.5:
        return 1  # Q1: Excited/Happy
    elif arousal >= 0.5 and valence < 0.5:
        return 2  # Q2: Angry/Anxious
    elif arousal < 0.5 and valence < 0.5:
        return 3  # Q3: Calm/Sad
    else:
        return 4  # Q4: Calm/Happy

def calculate_quadrant_accuracy(y_true, y_pred):
    """Calculate quadrant prediction accuracy"""
    return accuracy_score(y_true, y_pred), confusion_matrix(y_true, y_pred, labels=[1,2,3,4])

def plot_confusion_matrix(conf_matrix):
    """Visualize quadrant confusion matrix"""
    labels = ['Q1: Happy/Excited', 'Q2: Angry/Anxious', 'Q3: Sad/Calm', 'Q4: Happy/Calm']
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(conf_matrix, cmap='Blues')

    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(4):
        for j in range(4):
            ax.text(j, i, conf_matrix[i, j],
                    ha="center", va="center",
                    color="white" if conf_matrix[i,j] > conf_matrix.max()/2 else "black")

    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Emotion Quadrant Confusion Matrix')
    plt.tight_layout()
    return fig

def extract_midi_features(midi_file):
    """Extract musical features from MIDI files"""
    try:
        midi = pretty_midi.PrettyMIDI(midi_file)
        if not midi.instruments:
            return None

        all_notes = []
        for inst in midi.instruments:
            all_notes.extend(inst.notes)

        if len(all_notes) < 10:  # Skip files with few notes
            return None

        # Calculate features
        pitches = [n.pitch for n in all_notes]
        velocities = [n.velocity for n in all_notes]
        durations = [n.end-n.start for n in all_notes]
        intervals = np.diff(sorted([n.start for n in all_notes]))

        return {
            'pitch_mean': np.mean(pitches),
            'pitch_std': np.std(pitches),
            'velocity_mean': np.mean(velocities),
            'duration_mean': np.mean(durations),
            'interval_std': np.std(intervals) if len(intervals) > 0 else 0,
            'note_density': len(all_notes)/midi.get_end_time()
        }
    except Exception as e:
        print(f"Error processing {midi_file}: {str(e)}")
        return None

def calculate_vendi_score(features_list):
    """Calculate diversity score using PCA and entropy"""
    df = pd.DataFrame([f for f in features_list if f is not None])
    if len(df) < 2:
        return 0.0

    # Preprocess features
    df = df.fillna(0)
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)

    # Dimensionality reduction
    pca = PCA(n_components=min(3, len(df.columns), len(df)))
    reduced = pca.fit_transform(scaled)

    # Calculate normalized entropy
    entropies = []
    for dim in range(reduced.shape[1]):
        hist = np.histogram(reduced[:,dim], bins=10)[0]
        hist = hist/hist.sum()
        entropies.append(entropy(hist)/np.log(len(hist)))

    return np.mean(entropies)

def compare_original_vs_generated(original, generated):
    """Main comparison logic"""
    # Filter out None values
    orig_features = [extract_midi_features(p) for p,_,_ in original]
    gen_features = [extract_midi_features(p) for p,_,_ in generated]
    orig_features = [f for f in orig_features if f is not None]
    gen_features = [f for f in gen_features if f is not None]

    # Calculate metrics
    vendi_orig = calculate_vendi_score(orig_features)
    vendi_gen = calculate_vendi_score(gen_features)

    # Quadrant accuracy
    y_true = [get_emotion_quadrant(v,a) for _,v,a in original]
    y_pred = [get_emotion_quadrant(v,a) for _,v,a in generated]
    accuracy, cm = calculate_quadrant_accuracy(y_true, y_pred)

    # Save confusion matrix
    fig = plot_confusion_matrix(cm)
    fig.savefig('confusion_matrix.png')
    plt.close(fig)

    return {
        'quadrant_accuracy': accuracy,
        'vendi_ratio': vendi_gen/vendi_orig if vendi_orig > 0 else 0,
        'original_vendi': vendi_orig,
        'generated_vendi': vendi_gen
    }

# Data Loading and Processing
def parse_emopia_filename(filename):
    """Parse EMOPIA quadrant from filename"""
    try:
        if filename.startswith('Q') and filename[1].isdigit():
            return int(filename[1])
    except:
        return None

def load_datasets():
    """Load and validate both datasets"""
    # Load EMOPIA files
    emopia_files = [f for f in os.listdir(EMOPIA_DIR) if f.endswith('.mid')]
    emopia_sample = np.random.choice(emopia_files, SAMPLE_SIZE, replace=False)

    original = []
    for f in emopia_sample:
        quad = parse_emopia_filename(f)
        if quad in [1,2,3,4]:
            original.append((
                os.path.join(EMOPIA_DIR, f),
                *{1: (0.75,0.75), 2: (0.25,0.75), 3: (0.25,0.25), 4: (0.75,0.25)}[quad]
            ))

    # Load generated files
    generated = []
    for f in os.listdir(GENERATED_DIR)[:SAMPLE_SIZE]:
        if f.endswith('.mid'):
            try:
                parts = f.split('_')
                v = float(parts[0][1:])
                a = float(parts[1][1:])
                generated.append((os.path.join(GENERATED_DIR, f), v, a))
            except:
                continue

    return original[:SAMPLE_SIZE], generated[:SAMPLE_SIZE]

# Main Execution
if __name__ == "__main__":
    original, generated = load_datasets()

    if len(original) < 10 or len(generated) < 10:
        raise ValueError("Insufficient valid files for analysis")

    results = compare_original_vs_generated(original, generated)

    print(f"\n{' Analysis Results ':=^40}")
    print(f"Quadrant Accuracy: {results['quadrant_accuracy']:.2%}")
    print(f"VENDI Score Ratio: {results['vendi_ratio']:.2f}")
    print(f"Original VENDI: {results['original_vendi']:.3f}")
    print(f"Generated VENDI: {results['generated_vendi']:.3f}")
    print(f"\nConfusion matrix saved to: confusion_matrix.png")


=========== Analysis Results ===========
Quadrant Accuracy: 28.57%
VENDI Score Ratio: 1.06
Original VENDI: 0.848
Generated VENDI: 0.903

Confusion matrix saved to: confusion_matrix.png
